In [ ]:
!pip install langchain 
!pip install gpt4all 
!pip install faiss-cpu 
!pip install huggingface-hub 
!pip install sentence-transformers 

In [ ]:
from langchain.document_loaders import PyPDFLoader
from langchain import PromptTemplate
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores.faiss import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain.llms import GPT4All

In [ ]:
documents = PyPDFLoader('./LocalDataForTraining/Invoice1.pdf').load_and_split()
documents

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1024,
                                               chunk_overlap = 64)
texts = text_splitter.split_documents(documents)
texts

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')
faiss_index = FAISS.from_documents(texts, embeddings)
faiss_index.save_local("./index")

In [ ]:
faiss_index.save_local("./index")

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-MiniLM-L6-v2')
faiss_index = FAISS.load_local("./index", embeddings)

In [ ]:
from gpt4all import GPT4All
llm = GPT4All("mistral-7b-openorca.Q4_0.gguf")

In [ ]:
from gpt4all import GPT4All
GPT4All.list_models()

In [ ]:
# load vector store index
embeddings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')
faiss_index = FAISS.load_local("./index", embeddings)

In [ ]:
from langchain.llms import GPT4All
llm = GPT4All(model='mistral-7b-openorca.Q4_0.gguf')

In [ ]:
template = """
Please use the following context to answer the question concisely and without including the context in your answer.
Context: {context}
Question: {question}
Answer: 
"""

In [ ]:
def ask_question(question):
    # retrieves the top 4 most similar documents based on the question
    matched_docs = faiss_index.similarity_search(question, 4)

    context = ""
    # append all the matched documents
    for doc in matched_docs:
        context += doc.page_content + " \n\n"

    # create the prompt template and pass in the context variable
    prompt = PromptTemplate(template = template, 
        input_variables=["context", "question"]).partial(
            context = context)

    # creates the chain   
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"question": question})    

In [ ]:
while True:
    print(ask_question(input('Question: ')))

In [ ]:
import os
pdf_folder_path = "./LocalDataForTraining/"
pdf_dir = os.listdir(pdf_folder_path)

# for macOS only
pdf_dir.remove('.DS_Store')

# Load multiple files
loaders = [PyPDFLoader(os.path.join(pdf_folder_path, fn))
              for fn in pdf_dir]

all_documents = []

for loader in loaders:
    pdf_documents = loader.load_and_split() # .load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1024,
                                               chunk_overlap = 64)
    documents = text_splitter.split_documents(pdf_documents)
    all_documents.extend(documents)

embeddings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')

faiss_index = FAISS.from_documents(all_documents, embeddings)
faiss_index.save_local("./index")

In [ ]:
import os

pdf_folder_path = "./LocalDataForTraining/"
pdf_dir = os.listdir(pdf_folder_path)

pdf_dir.remove('.DS_Store')  
loaders = [PyPDFLoader(os.path.join(pdf_folder_path, fn))      
              for fn in pdf_dir]  

In [ ]:
all_documents = []

for loader in loaders:
    documents = loader.load_and_split() 
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1024,
                                                   chunk_overlap = 64)
    documents = text_splitter.split_documents(documents)
    all_documents.extend(documents)

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-MiniLM-L6-v2')

faiss_index = FAISS.from_documents(all_documents, embeddings)
faiss_index.save_local("./index")

In [ ]:
template = """
Please use the following context to answer the question concisely and without including the context in your answer.
Context: {context}
Question: {question}
Answer: 
"""

def ask_question(question):
    matched_docs = faiss_index.similarity_search(question, 4)  
    context = ""
    for doc in matched_docs:  
        context += doc.page_content + " \n\n"
    prompt = PromptTemplate(template = template,  
        input_variables=["context", "question"]).partial(
            context = context)
    chain = prompt | llm | StrOutputParser()  
    return chain.invoke({"question": question})

while True:
    print(ask_question(input('Question: ')))

In [ ]:
from langchain.document_loaders import CSVLoader
documents = CSVLoader('./Titanic_train.csv').load_and_split()
documents

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1024,
                                               chunk_overlap = 64)
texts = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(
                 model_name = 'sentence-transformers/all-MiniLM-L6-v2')
faiss_index = FAISS.from_documents(texts, embeddings)

def ask_question(question):  
    matched_docs = faiss_index.similarity_search(question, 4)  

    context = ""
    for doc in matched_docs:  
        context += doc.page_content + " \n\n"

    prompt = PromptTemplate(template = template,  
        input_variables=["context", "question"]).partial(
            context = context)

    chain = prompt | llm | StrOutputParser()  
    return chain.invoke({"question": question})

In [ ]:
template = """
Please use the following context to answer the question concisely and without including the context in your answer.
Context: {context}
Question: {question}
Answer: 
"""

def ask_question(question):
    # retrieves the top 4 most similar documents based on the question
    matched_docs = faiss_index.similarity_search(question, 4)

    context = ""
    # append all the matched documents
    for doc in matched_docs:
        context += doc.page_content + " \n\n"

    # create the prompt template and pass in the context variable
    prompt = PromptTemplate(template = template, 
        input_variables=["context", "question"]).partial(
            context = context)

    # creates the chain   
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"question": question})    

In [ ]:
while True:
    print(ask_question(input('Question: ')))

In [ ]:
!pip install jq

In [ ]:
from langchain.document_loaders import JSONLoader

documents = JSONLoader('./nobel_laureates.json',
                       jq_schema='.laureates[]',
                       text_content=False).load_and_split()
documents

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1024,
                                               chunk_overlap = 64)
texts = text_splitter.split_documents(documents)

# use the model to convert input text into dense numerical vectors
embeddings = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-MiniLM-L6-v2')

# builds an index to quickly retrieve similar documents
faiss_index = FAISS.from_documents(texts, embeddings)

template = """
Please use the following context to answer the question concisely and without including the context in your answer.
Context: {context}
Question: {question}
Answer: 
"""

def ask_question(question):
    # retrieves the top 4 most similar documents based on the question
    matched_docs = faiss_index.similarity_search(question, 4)

    context = ""
    # append all the matched documents
    for doc in matched_docs:
        context += doc.page_content + " \n\n"

    # create the prompt template and pass in the context variable
    prompt = PromptTemplate(template = template, 
        input_variables=["context", "question"]).partial(
            context = context)

    # creates the chain   
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"question": question})    

while True:    
    print(ask_question(input('Question: ')))

In [ ]:
import pandas as pd

df = pd.read_json('famous_people.json')
df

In [ ]:
{
  "famous_people": [
    {
      "name": "John Smith",
      "occupation": "Actor",
      "birth_date": "1980-05-15",
      "birth_place": "Los Angeles, USA",
      "achievements": ["Oscar-winning performance", "Golden Globe nominee"]     
    },
    {
      "name": "Emily Johnson",
      "occupation": "Tech Entrepreneur",
      "birth_date": "1985-02-20",
      "birth_place": "San Francisco, USA",
      "achievements": ["Founder of Tech Innovations Inc.", "Forbes 30 Under 30"]      
    },
    {
      "name": "Carlos Rodriguez",
      "occupation": "Chef",
      "birth_date": "1972-09-08",
      "birth_place": "Barcelona, Spain",
      "achievements": ["Michelin Star Chef", "Best-selling cookbook author"]      
    },
    {
      "name": "Aisha Patel",
      "occupation": "Humanitarian",
      "birth_date": "1988-11-30",
      "birth_place": "Mumbai, India",
      "achievements": ["Founder of AidGlobal Foundation", "UNICEF Ambassador"]      
    },
    {
      "name": "Yuki Tanaka",
      "occupation": "Fashion Designer",
      "birth_date": "1983-03-10",
      "birth_place": "Tokyo, Japan",
      "achievements": ["International Fashion Award", "Creative Director of Vogue Japan"]      
    },
    {
      "name": "Isabella Martinez",
      "occupation": "Explorer",
      "birth_date": "1982-08-12",
      "birth_place": "Madrid, Spain",
      "achievements": ["Discovered ancient ruins in South America", "National Geographic Explorer of the Year"]
    },
    {
      "name": "Liam Johnson",
      "occupation": "Astronaut",
      "birth_date": "1987-04-25",
      "birth_place": "Houston, USA",
      "achievements": ["Mission Commander on Mars Expedition", "NASA Medal of Honor"]
    },
    {
      "name": "Sophia Nguyen",
      "occupation": "Environmental Scientist",
      "birth_date": "1985-11-03",
      "birth_place": "Hanoi, Vietnam",
      "achievements": ["Published groundbreaking research on sustainable agriculture", "Recipient of Green Earth Award"]
    },
    {
      "name": "Noah Thompson",
      "occupation": "Inventor",
      "birth_date": "1990-02-18",
      "birth_place": "Sydney, Australia",
      "achievements": ["Patented revolutionary renewable energy device", "Tech Innovator of the Year"]
    },
    {
      "name": "Olivia Patel",
      "occupation": "Classical Pianist",
      "birth_date": "1989-07-09",
      "birth_place": "Mumbai, India",
      "achievements": ["Performed at prestigious concert halls worldwide", "Grammy Award for Best Classical Performance"]
    }
  ]
}

In [ ]:
import json
import pandas as pd
from pandas import json_normalize

with open('famous_people.json', 'r') as json_file:
    json_data = json.load(json_file)

# load JSON data into Pandas DataFrame
df = json_normalize(json_data, 'famous_people')
df

In [ ]:
from langchain.llms import GPT4All

model = 'mistral-7b-openorca.Q4_0.gguf'
llm = GPT4All(model = model)

In [ ]:
template = """
    Here is schema of a Pandas DataFrame (df):
    name,occupation,birth_date,birth_place,achievements
    I will start prompting you and you must return the response 
    as a single Python statement so that I can execute it the 
    result using the eval() function. 
    
    For your info I have loaded the JSON file as a df using the
    following code:
    
    with open('famous_people.json', 'r') as json_file:
    json_data = json.load(json_file)

    # load JSON data into Pandas DataFrame
    df = json_normalize(json_data, 'famous_people')

    Question: {question}
"""

In [ ]:
from langchain import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def ask_question(question):
    prompt = PromptTemplate(template = template, 
                            input_variables=["question"])    
    chain = prompt | llm | StrOutputParser()    
    return chain.invoke({"question": question})

In [ ]:
while True:    
    print(ask_question(input('Question: ')))

In [ ]:
import json
import pandas as pd
from pandas import json_normalize

with open('famous_people.json', 'r') as json_file:
    json_data = json.load(json_file)

# load JSON data into Pandas DataFrame
df = json_normalize(json_data, 'famous_people')
df

In [ ]:
messages = []
messages.append(
{
    'role':'user',
    'content':'''
        Here is an example of a JSON file loaded into a Pandas DataFrame:
        {
          "famous_people": [
            {
              "name": "John Smith",
              "occupation": "Actor",
              "birth_date": "1980-05-15",
              "birth_place": "Los Angeles, USA",
              "achievements": ["Oscar-winning performance", 
                               "Golden Globe nominee"],
              "quote": "Acting is not about being someone different. 
                        It's finding the similarity in what is 
                        apparently different, then finding myself in
                        there."
            },
          ]
        }
        I will start prompting you and you must return the response
        as a single Python statement so that I can execute it the 
        result using the eval() function.

        For your info I have loaded the JSON file as a df using the following code:
    
        with open('famous_people.json', 'r') as json_file:
        json_data = json.load(json_file)
    
        # load JSON data into Pandas DataFrame
        df = json_normalize(json_data, 'famous_people')
    '''
})  

In [ ]:
from openai import OpenAI
import re
import os

os.environ['OPENAI_API_KEY'] = ""

client = OpenAI(
    api_key = os.environ.get("OPENAI_API_KEY"),
)

while True:
    prompt = input('\nAsk a question: ') 
    if prompt == "quit":
        break
        
    messages.append(
    {
        'role':'user',
        'content':prompt
    })    

    completion = client.chat.completions.create(
        model = "gpt-4o",
        messages = messages,
        max_tokens = 1024,
        temperature = 0)
    
    response = completion.choices[0].message.content   
    
    pattern = re.compile(r'```python\s*([\s\S]*)\n```')
    match = pattern.search(response)

    if match:
        extracted_content = match.group(1)
        print(extracted_content)
        if extracted_content.count('\n') > 1:
            exec(extracted_content)   # use this for plotting    
        else:              
            display(eval(extracted_content))  # use this for query        
    else:
        print("No content found within ```python...```.")
    
    messages.append(
    {
        'role':'assistant',
        'content':response
    })